# Optional: scoring and duplicate embeddings

In [ ]:
import ast

import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

np.random.seed(42)



# Scoring

In [2]:
def score_model(y_true, y_pred, print_results=True):
    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)
    # F1 score
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    scores = {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "f1_weighted": f1_weighted,
    }

    if print_results:
        print(f"Accuracy: {accuracy:.4f}")
        print(f"F1 macro: {f1_macro:.4f}")
        print(f"F1 micro: {f1_micro:.4f}")
        print(f"F1 weighted: {f1_weighted:.4f}")

    return scores

In [3]:
def confidence_score(y_true, y_pred, confidence):
    """
    Score for the confidence weighted prediction task.

    Parameters:
    -----------
    y_true : array-like
        True class labels
    y_pred : array-like
        Predicted class labels
    confidence : array-like
        Confidence for each prediction, in [0, 1]
        (= estimated probability that the prediction is correct)

    Returns:
    --------
    float
        weighted F1 * (1 - Brier score), between 0 and 1 (higher is better)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence, dtype=float)

    assert len(y_true) == len(y_pred) == len(confidence), "Inputs must have the same length"
    assert np.all((confidence >= 0) & (confidence <= 1)), "Confidence must be between 0 and 1"

    f1 = f1_score(y_true, y_pred, average="weighted")
    correct = (y_true == y_pred).astype(float)
    brier = np.mean((confidence - correct) ** 2)

    return f1 * (1 - brier)

# Loading Data

In [4]:
df = pd.read_pickle("../../../data/df_train.pkl")
df_train, df_val = train_test_split(df, test_size=0.1, random_state=42, stratify=df["industry"])
df_train["business_description_embedding"] = df_train["business_description_embedding"].apply(ast.literal_eval)
df_val["business_description_embedding"] = df_val["business_description_embedding"].apply(ast.literal_eval)

X_train = np.array(df_train["business_description_embedding"].tolist())
X_val = np.array(df_val["business_description_embedding"].tolist())

label_encoder = LabelEncoder().fit(df_train.industry)

y_train = label_encoder.transform(df_train.industry)
y_val = label_encoder.transform(df_val.industry)

# Preprocess Data

In [5]:
def remove_duplicates(df):
    # Count exact duplicates
    exact_duplicates = df.business_description_embedding.duplicated().sum()
    print(f"Number of exact duplicate embeddings: {exact_duplicates}")

    df = df.drop_duplicates(subset=["business_description_embedding"])
    return df

df_train = remove_duplicates(df_train)

# Create embeddings and labels for this reduced dataset
X_train = np.array(df_train["business_description_embedding"].tolist())
y_train = label_encoder.transform(df_train.industry)

# val set
df_val = remove_duplicates(df_val)
X_val = np.array(df_val["business_description_embedding"].tolist())
y_val = label_encoder.transform(df_val.industry)

Number of exact duplicate embeddings: 4124
Number of exact duplicate embeddings: 442


## Continue from Preprocessed data

In [8]:
# Save the preprocessed data for quick loading
df_train.to_pickle("../../data/processed/df_train_preprocessed.pkl")
df_val.to_pickle("../../data/processed/df_val_preprocessed.pkl")


In [9]:
# Reload the preprocessed data
df_train = pd.read_pickle("../../data/processed/df_train_preprocessed.pkl")
df_val = pd.read_pickle("../../data/processed/df_val_preprocessed.pkl")

X_train = np.array(df_train["business_description_embedding"].tolist())
y_train = label_encoder.transform(df_train.industry)


## Ensemble System

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bagging_clf = BaggingClassifier(
    n_estimators=50,
    estimator=DecisionTreeClassifier(),
    random_state=42,
    n_jobs=-1
)
bagging_clf.fit(X_train, y_train)

y_probs = bagging_clf.predict_proba(X_val)
y_pred = bagging_clf.classes_[np.argmax(y_probs, axis=1)]
y_proba = np.max(y_probs, axis=1)
print(f"Score with Bagging: {confidence_score(y_val, y_pred, y_proba):.4f}")


Score with Bagging: 0.5413


In [ ]:
from skopt import BayesSearchCV
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='logloss')


search_spaces = {
    'learning_rate': (0.01, 1.0, 'log-uniform'),
    'n_estimators': (50, 500),
    'max_depth': (1, 50),
    'subsample': (0.1, 1.0, 'uniform'), # fraction of samples used for each tree
    'colsample_bytree': (0.1, 1.0, 'uniform') # fraction of features used for each tree
}

In [ ]:
# Setup BayesSearchCV
opt = BayesSearchCV(
    estimator=xgb,
    search_spaces=search_spaces,
    n_iter=32,
    cv=3,
    random_state=42,
    n_jobs=-1
)

# Perform hyperparameter optimization
opt.fit(X_train, y_train)

# Best parameters found
print(f"Best Hyperparameters: {opt.best_params_}")
print(f"Best Score: {opt.best_score_}")

In [ ]:
# save xgboost model 
opt.best_estimator_.save_model("xgb_model.json")

xgb = opt.best_estimator_

def load_xgb_model():
    # load xgboost model 
    xgb = XGBClassifier()
    xgb.load_model("xgb_model.json")
    return xgb

# if trained before 
# xgb = load_xgb_model()

# predict
y_pred = xgb.predict(X_val)
y_probs = xgb.predict_proba(X_val)

# score
print(f"Score with XGBoost: {confidence_score(y_val, y_pred, y_probs):.4f}")
score_model(y_val, y_pred, y_probs)


In [ ]:
n_industries = len(np.unique(y_train))
industry_cmap = ListedColormap(cc.glasbey[:n_industries])


In [ ]:
def flag_uncertain_predictions_for_human_review(X_val, confidence, y_true, threshold=0.6, visualize=True, per_class=False): 
    # flag predictions with confidence below threshold
    uncertain_predictions = confidence < threshold
    X_val_uncertain = X_val[uncertain_predictions]
    y_probs_uncertain = confidence[uncertain_predictions]
    y_true_uncertain = y_true[uncertain_predictions]

    uncertain_df = pd.DataFrame({
        "business_description_embedding": X_val_uncertain,
        "confidence": y_probs_uncertain,
        "true_industry": y_true_uncertain
    })

    if visualize and not per_class:
        # Show which predictions are uncertain 
        # overlay all predictions pca with low alpha with the uncertain predictions more visible 
        
        plt.title("PCA Uncertain Predictions")
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_val)
        plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.1)
        plt.scatter(X_pca[uncertain_predictions, 0], X_pca[uncertain_predictions, 1], alpha=1, c=y_true, cmap=industry_cmap)
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.show()
    elif visualize and per_class:

    return uncertain_df


flag_uncertain_predictions_for_human_review(X_val, y_probs, y_val, visualize=True)

